# 4.3 Data Preprocessing and Data Representation

**News Simplification Assistant — Data Preparation Notebook**

This notebook prepares the XSum dataset for prompting `facebook/opt-125m`.
Each numbered section below corresponds to one step of the preprocessing
pipeline described in Section 4.3 of the report:

1. Download the XSum dataset and save a raw copy to `data/raw`
2. Load selected XSum articles and reference summaries
3. Remove empty or duplicated articles
4. Clean newline characters, extra spaces and unnecessary symbols
5. Split articles exceeding ~700 words into sentence-based chunks
6. Insert each chunk into a fixed prompt template
7. Tokenise each chunk prompt using the OPT tokenizer

If `data/raw` is empty, the notebook downloads XSum from the Hugging Face
Hub first. The final processed dataset (cleaned articles + chunks +
prompts + token ids) is written to `data/processed` in the last cell.

**Why chunking?** `facebook/opt-125m` has a 2048-token context window,
but a single fixed prompt built from a long article can exceed that by a
wide margin (e.g. 4421 tokens seen for the longest article) — truncating
it would silently cut off context mid-sentence. Instead, long articles
are split into multiple sentence-respecting chunks; each chunk is
simplified by the model independently, and the resulting chunk summaries
are then combined and fed back into the model once more to produce the
final Summary / Key Points / Vocabulary / Quiz output (done at generation
time in `opt-125m.ipynb`).


## 0. Setup — Imports and Paths

In [1]:
import os
import re
import json
import glob
import random

import pandas as pd

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

random.seed(42)

print(f"Raw data directory:       {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")


Raw data directory:       data/raw
Processed data directory: data/processed


## 1. Download the XSum Dataset to `data/raw`

`data/raw` does not contain the XSum dataset yet, so this step downloads
it from the Hugging Face Hub (`EdinburghNLP/xsum`) and writes a raw JSONL
copy to `data/raw/xsum_raw.jsonl`.

- If a raw file already exists, the download is skipped (safe to re-run).
- `XSUM_SAMPLE_SIZE` controls how many articles are pulled for this
  project so downstream steps stay fast; set it to `None` to download
  the full dataset (~226k train articles).
- Requires internet access to `huggingface.co`. If this environment has
  no internet access, run this cell once on a machine that does, then
  copy the resulting `data/raw/xsum_raw.jsonl` file over.


In [2]:
from datasets import load_dataset

XSUM_SAMPLE_SIZE = 1500
XSUM_SEED = 42



os.makedirs(RAW_DIR, exist_ok=True)
output_path = os.path.join(RAW_DIR, "xsum_raw.jsonl")

if os.path.exists(output_path):
    print(f"Raw XSum file already exists at '{output_path}', skipping download.")
    
else:
    print("Downloading XSum dataset from the Hugging Face Hub (EdinburghNLP/xsum)...")
    dataset = load_dataset("EdinburghNLP/xsum", split="train")

    if XSUM_SAMPLE_SIZE is not None and XSUM_SAMPLE_SIZE < len(dataset):
        dataset = dataset.shuffle(seed=XSUM_SEED).select(range(XSUM_SAMPLE_SIZE))

        with open(output_path, "w", encoding="utf-8") as f:
            for row in dataset:
                record = {
                    "id": row.get("id"),
                    "document": row.get("document"),
                    "summary": row.get("summary"),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"Saved {len(dataset)} raw XSum articles to '{output_path}'")

Raw XSum file already exists at 'data/raw\xsum_raw.jsonl', skipping download.


## 2. Load Selected XSum Articles and Reference Summaries

The raw XSum data downloaded in Section 1 is read back from `data/raw`.
This loader supports a few common formats so it works regardless of how
the raw dump was saved:

- A HuggingFace `datasets` folder saved with `save_to_disk()`
- A single `.json` / `.jsonl` file (list of records or one record per line)
- A `.csv` file

Each record is expected to contain an **article** field and a **reference
summary** field (XSum uses `document` and `summary`).


In [3]:
json_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.jsonl")))
if json_files:
    records = []
    for fp in json_files:
        with open(fp, "r", encoding="utf-8") as f:
            if fp.endswith(".jsonl"):
                for line in f:
                    line = line.strip()
                    if line:
                        records.append(json.loads(line))
            else:
                data = json.load(f)
                records.extend(data if isinstance(data, list) else [data])

df = pd.DataFrame(records)

rename_map = {
    "document": "article",
    "summary": "summary",
    "id": "article_id",
}
raw_df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print(f"Loaded {len(raw_df)} raw articles from '{RAW_DIR}'")
raw_df.head()


Loaded 1500 raw articles from 'data/raw'


,article_id,article,summary
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...


## 3. Remove Empty or Duplicated Articles

Records with an empty article/summary, or with an article that is a
duplicate of one already seen, are dropped.


In [4]:
def remove_empty_and_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)

    # Drop missing values
    df = df.dropna(subset=["article", "summary"])

    # Drop empty / whitespace-only strings
    df = df[df["article"].str.strip().astype(bool)]
    df = df[df["summary"].str.strip().astype(bool)]

    # Drop exact duplicate articles, keep the first occurrence
    df = df.drop_duplicates(subset=["article"], keep="first")

    df = df.reset_index(drop=True)

    after = len(df)
    print(f"Removed {before - after} empty/duplicate articles ({before} -> {after})")
    return df


dedup_df = remove_empty_and_duplicates(raw_df)
dedup_df.head()


Removed 1 empty/duplicate articles (1500 -> 1499)


,article_id,article,summary
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...


## 4. Clean Newline Characters, Extra Spaces and Unnecessary Symbols

Basic text normalisation applied to both the article and the reference
summary:

- Collapse `\n`, `\r`, `\t` into single spaces
- Collapse repeated whitespace into a single space
- Strip stray control characters and unusual unicode artefacts
- Trim leading/trailing whitespace


In [5]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalise newlines / tabs to spaces
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove control characters
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)

    # Remove stray repeated punctuation/symbols e.g. "----", "****"
    text = re.sub(r"[-*_=~]{3,}", " ", text)

    # Collapse multiple spaces into one
    text = re.sub(r"\s+", " ", text)

    return text.strip()


clean_df = dedup_df.copy()
clean_df["article"] = clean_df["article"].apply(clean_text)
clean_df["summary"] = clean_df["summary"].apply(clean_text)

# Cleaning can occasionally expose new empty strings - drop those too
clean_df = clean_df[clean_df["article"].astype(bool)].reset_index(drop=True)

print(f"{len(clean_df)} articles remaining after cleaning")
clean_df.head()


1499 articles remaining after cleaning


,article_id,article,summary
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...


## 5. Split Articles Exceeding ~700 Words into Sentence-based Chunks

`facebook/opt-125m` has a 2048-token context window. A single fixed
4-section prompt built from a long article plus the instructions can
exceed that comfortably (e.g. 4421 tokens for the longest article seen
here), which would truncate the article mid-sentence and cut off context
the model needs.

Instead of truncating, articles longer than `MAX_CHUNK_WORDS` (700 words)
are split into multiple **chunks**, breaking only on full sentence
boundaries so no sentence is ever cut in half. Every article's chunks are
stored in a new `chunk` column as a **list of strings** — the appropriate
data type here, since an article may need anywhere from 1 to several
chunks. Short articles simply get a `chunk` list of length 1.


In [6]:
MAX_CHUNK_WORDS = 700  # articles longer than this are split into multiple sentence-based chunks

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str):
    """Split text into sentences on '.', '!', '?' followed by whitespace."""
    sentences = SENTENCE_SPLIT_RE.split(text.strip())
    return [s for s in sentences if s]


def split_into_chunks(text: str, max_words: int = MAX_CHUNK_WORDS):
    """Split article text into a list of chunks, each up to `max_words`
    words, breaking only on full sentence boundaries (never mid-sentence)."""
    sentences = split_sentences(text)
    chunks, current_sentences, current_words = [], [], 0

    for sentence in sentences:
        sentence_words = len(sentence.split())
        if current_sentences and current_words + sentence_words > max_words:
            chunks.append(" ".join(current_sentences))
            current_sentences, current_words = [], 0
        current_sentences.append(sentence)
        current_words += sentence_words

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks if chunks else [text]


chunked_df = clean_df.copy()
chunked_df["article_word_count"] = chunked_df["article"].str.split(" ").apply(len)
chunked_df["chunk"] = chunked_df["article"].apply(split_into_chunks)
chunked_df["chunk_count"] = chunked_df["chunk"].apply(len)

n_multi = int((chunked_df["chunk_count"] > 1).sum())
print(f"{n_multi} of {len(chunked_df)} articles were split into multiple chunks "
      f"(threshold: {MAX_CHUNK_WORDS} words)")
print("Chunk count distribution:")
print(chunked_df["chunk_count"].value_counts().sort_index())
chunked_df[["article_id", "article_word_count", "chunk_count", "chunk"]].head()


184 of 1499 articles were split into multiple chunks (threshold: 700 words)
Chunk count distribution:
chunk_count
1    1315
2     170
3      13
6       1
Name: count, dtype: int64


,article_id,article_word_count,chunk_count,chunk
0,35821725,548,1,"[In Wales, councils are responsible for fundin..."
1,40302059,210,1,"[Up to 100,000 youngsters will be eligible for..."
2,36192574,670,1,[Middlesbrough and Brighton face each other on...
3,32831331,562,1,"[Money allocated to children's centres, which ..."
4,26781682,337,1,[Both sides have overlapping claims in the sea...


## 6. Insert Each Chunk into a Prompt Template

Two fixed prompt templates are used:

- **`CHUNK_PROMPT_TEMPLATE`** — applied to every chunk in the `chunk`
  column, asking the model for a short, simplified summary of just that
  part of the article. The dataset's `prompt` column is built from this
  template — one prompt per chunk, so `prompt` is a **list of strings**
  the same length as `chunk`.
- **`FINAL_PROMPT_TEMPLATE`** — combines the simplified chunk summaries
  (generated later, at inference time) and asks the model to produce the
  final four labelled sections: Summary, Key Points, Vocabulary, Quiz.
  It can't be filled in here because it depends on model output that
  doesn't exist until generation time, so it's saved to
  `data/processed/prompt_templates.json` for `opt-125m.ipynb` to load
  and use once the chunk summaries have been generated.


In [7]:
CHUNK_PROMPT_TEMPLATE = """You are a news simplification assistant. Read the following part of a news article and write a short, simplified summary of just this part using clear and simple English. Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated in the article.

Article part:
{chunk}

Response:
Simplified summary:
"""

FINAL_PROMPT_TEMPLATE = """You are a news simplification assistant. Below are simplified summaries of consecutive parts of the same news article, in order. Combine them and rewrite the article as structured, easy-to-understand learning content with exactly four labelled sections, in this order:
(1) Summary: a short, simplified summary of the whole article.
(2) Key Points: 3-5 bullet points with the most important facts.
(3) Vocabulary: 3-5 specialised words from the article with simple definitions.
(4) Quiz Questions: 2-3 questions based on the article.
Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated.

Simplified part summaries:
{chunk_summaries}

Response:
Summary:
"""

SINGLE_PROMPT_TEMPLATE = """You are a news simplification assistant. Simplify the following news article for students using clear and simple English. Generate the following sections:
(1) Summary: a short, simplified summary of the article.
(2) Key Points: 3-5 bullet points with the most important facts.
(3) Vocabulary: 3-5 specialized words from the article with simple definitions.
(4) Quiz Questions: 2-3 questions based on the article.
Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated in the article. Ensure all generated content is factually consistent with the original article.

Article:
{article}

Response:
Summary:
"""


def build_chunk_prompts(chunks):
    return [CHUNK_PROMPT_TEMPLATE.format(chunk=chunk) for chunk in chunks]

def build_row_prompt(row):
    if row["chunk_count"] == 1:
        return [SINGLE_PROMPT_TEMPLATE.format(article=row["article"])]
    
    return build_chunk_prompts(row["chunk"])

prompt_df = chunked_df.copy()

# 2. THE FIX: Apply build_row_prompt across the rows (axis=1)
prompt_df["prompt"] = prompt_df.apply(build_row_prompt, axis=1)

print(f"Article 0 has {prompt_df.loc[0, 'chunk_count']} chunk(s). First prompt:\n")
print(prompt_df.loc[0, "prompt"][0])


Article 0 has 1 chunk(s). First prompt:

You are a news simplification assistant. Simplify the following news article for students using clear and simple English. Generate the following sections:
(1) Summary: a short, simplified summary of the article.
(2) Key Points: 3-5 bullet points with the most important facts.
(3) Vocabulary: 3-5 specialized words from the article with simple definitions.
(4) Quiz Questions: 2-3 questions based on the article.
Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated in the article. Ensure all generated content is factually consistent with the original article.

Article:
In Wales, councils are responsible for funding and overseeing schools. But in England, Mr Osborne's plan will mean local authorities will cease to have a role in providing education. Academies are directly funded by central government and head teachers have more freedom over admissions and to change th

## 7. Tokenise Each Chunk Prompt Using the OPT Tokenizer

Every chunk prompt (one per entry in the `prompt` list) is converted into
token ids using the tokenizer that ships with `facebook/opt-125m`. This
is the **data representation** step: prompt text becomes the numeric
input representation the model consumes, and generated output tokens
will later be decoded back into text using this same tokenizer.

Chunking in Section 5 already keeps each chunk prompt well under the
2048-token context window, but a token-level truncation is still applied
here as a safety net in case an unusually dense chunk still runs long.


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "facebook/opt-125m"
MAX_MODEL_TOKENS = 2048
MAX_PROMPT_TOKENS = 1800  # leaves headroom for the model's generated output

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_prompt(prompt: str):
    encoded = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )
    input_ids = encoded["input_ids"]
    return input_ids, len(input_ids)


def normalize_prompt_list(prompt_value):
    if isinstance(prompt_value, str):
        return [prompt_value]
    return list(prompt_value)


def tokenize_prompt_collection(prompt_value):
    prompts = normalize_prompt_list(prompt_value)
    input_ids_list, lengths = [], []
    for prompt in prompts:
        ids, length = tokenize_prompt(prompt)
        input_ids_list.append(ids)
        lengths.append(length)
    return input_ids_list, lengths


token_df = prompt_df.copy()
token_df["prompt_word_count"] = token_df["prompt"].apply(lambda ps: [len(p.split()) for p in normalize_prompt_list(ps)])
token_results = token_df["prompt"].apply(tokenize_prompt_collection)
token_df["input_ids"] = token_results.apply(lambda x: x[0])
token_df["prompt_token_length"] = token_results.apply(lambda x: x[1])

token_df["max_prompt_token_length"] = token_df["prompt_token_length"].apply(max)
filtered_df = token_df[token_df["max_prompt_token_length"] <= 800].reset_index(drop=True)
print(f"Filtered to {len(filtered_df)} rows with max chunk prompt length <= 800 tokens")

max_chunk_tokens = filtered_df["prompt_token_length"].apply(max).max()
print(f"Largest single chunk prompt after filtering: {max_chunk_tokens} tokens")
filtered_df[["article_id", "chunk_count", "prompt_token_length", "max_prompt_token_length"]].head()


Filtered to 1149 rows with max chunk prompt length <= 800 tokens
Largest single chunk prompt after filtering: 799 tokens


,article_id,chunk_count,prompt_token_length,max_prompt_token_length
0,35821725,1,[790],790
1,40302059,1,[424],424
2,26781682,1,[558],558
3,34606772,1,[368],368
4,40535348,1,[295],295


## 8. Assemble the Final Processed Dataset

Combine the cleaned article, reference summary, chunks, per-chunk
prompts, and tokenised representation into the final structured table
that will be saved for the fine-tuning / generation stage. `chunk`,
`prompt`, `prompt_word_count`, `prompt_token_length`, and `input_ids`
are all lists — one entry per chunk.


In [9]:
final_df = filtered_df[[
    "article_id",
    "article",
    "summary",
    "chunk",
    "chunk_count",
    "prompt",
    "prompt_word_count",
    "prompt_token_length",
    "input_ids",
]].reset_index(drop=True)

if len(final_df) < 1000:
    raise ValueError(
        f"Filtered dataset contains only {len(final_df)} rows; increase XSUM_SAMPLE_SIZE or relax the token limit."
    )

final_df = final_df.sample(n=1000, random_state=42).reset_index(drop=True)
print(f"Final processed dataset: {len(final_df)} rows (sampled exactly 1000 rows)")
final_df.head()


Final processed dataset: 1000 rows (sampled exactly 1000 rows)


,article_id,article,summary,chunk,chunk_count,prompt,prompt_word_count,prompt_token_length,input_ids
0,34577037,Sam Ward also found the net but two goals from...,Great Britain were beaten 3-2 in the first of ...,[Sam Ward also found the net but two goals fro...,1,[You are a news simplification assistant. Simp...,[189],[273],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
1,35794084,A blue Seat Leon car was involved in a collisi...,A teenage boy has died after a crash on the A9...,[A blue Seat Leon car was involved in a collis...,1,[You are a news simplification assistant. Simp...,[297],[412],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
2,20266195,"Soares, 26, joined the Potters in 2008 but mad...",Bury have signed former Stoke City midfielder ...,"[Soares, 26, joined the Potters in 2008 but ma...",1,[You are a news simplification assistant. Simp...,[222],[313],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
3,34551854,"This is ""to ensure it was thorough, properly c...",Another police force has been asked to review ...,"[This is ""to ensure it was thorough, properly ...",1,[You are a news simplification assistant. Simp...,[630],[780],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
4,34769588,Simon Lee Bell was taken to hospital after an ...,The family of a 22-year-old man who died follo...,[Simon Lee Bell was taken to hospital after an...,1,[You are a news simplification assistant. Simp...,[195],[268],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."


## 9. Save Processed Data to `data/processed`

The final dataset is saved as JSONL (one record per line) so it can be
loaded lazily for the generation/fine-tuning stage without pulling the
whole dataset into memory. The two prompt templates are also saved to
`data/processed/prompt_templates.json` so `opt-125m.ipynb` can reuse the
exact same wording — in particular `FINAL_PROMPT_TEMPLATE`, which is
applied later once chunk summaries have been generated.


In [10]:
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "xsum_processed.jsonl")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in final_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(final_df)} processed records to '{OUTPUT_PATH}'")

TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")
with open(TEMPLATES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "chunk_prompt_template": CHUNK_PROMPT_TEMPLATE,
            "final_prompt_template": FINAL_PROMPT_TEMPLATE,
            "single_prompt_template": SINGLE_PROMPT_TEMPLATE,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved prompt templates to '{TEMPLATES_PATH}'")


Saved 1000 processed records to 'data/processed\xsum_processed.jsonl'
Saved prompt templates to 'data/processed\prompt_templates.json'
